In [3]:
from openai import OpenAI
from dotenv import load_dotenv
import os

In [4]:
load_dotenv() # Load environment variables from .env file
client=OpenAI(
    api_key=os.getenv("GROQ_API_KEY") ,# Ensure you have GROQ_API_KEY set in your .env file
    base_url="https://api.groq.com/openai/v1"
)

Semantic Chunking

In [5]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
import json
import os

def perform_direct_pdf_chunking(folder_path="documents/pdfdata"):
    embed_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    chunker=SemanticChunker(embed_model,breakpoint_threshold_type="percentile")
    all_smart_chunks=[]
    print(f"{folder_path} contains {len(os.listdir(folder_path))} files.")
    for filename in os.listdir(folder_path):
        if filename.endswith(".pdf"):
            loader=PyPDFLoader(os.path.join(folder_path,filename))
            docs=loader.load()
            chunks=chunker.split_documents(docs)
            all_smart_chunks.extend(chunks)
            print(f"{filename} split into {len(chunks)} smart chunks:")
    final_data=[]
    for chunk in all_smart_chunks:
        final_data.append({
            "metadata": chunk.metadata,
            "content": chunk.page_content
        })
    with open("documents/semantic_chunks_with_metadata.json","w") as fp:
        json.dump(final_data,fp,indent=4)
perform_direct_pdf_chunking()

e:\AgenticRagProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\prana\AppData\Local\Temp\ipykernel_22884\906600882.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embed_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5059.12it/s]


documents/pdfdata contains 1 files.
Constitution_India.pdf split into 867 smart chunks:


In [6]:
import json

# 1. Load the file
with open("documents/semantic_chunks_with_metadata.json", "r") as f:
    sample_data = json.load(f)

# 2. Check the first item
first_item = sample_data[len(sample_data)-1]

print("DATA STRUCTURE INSPECTION:")
print(f"Variable Type: {type(first_item)}")
print(f"Available Keys: {first_item.keys()}")

# 3. Access the content safely
# If your key is actually 'page_content' or something else, it will show up here
for key in first_item.keys():
    print(f"--- PREVIEW OF '{key}': ---")
    print(str(first_item[key])[:1000])

DATA STRUCTURE INSPECTION:
Variable Type: <class 'dict'>
Available Keys: dict_keys(['metadata', 'content'])
--- PREVIEW OF 'metadata': ---
{'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2024-07-01T11:20:33+00:00', 'source': 'documents/pdfdata\\Constitution_India.pdf', 'total_pages': 402, 'page': 401, 'page_label': '402'}
--- PREVIEW OF 'content': ---
562(E), dated the 6th August, 2019, Gazette of India, Extraordinary, Part II, Section 3, Sub-section (i).


In [9]:
import json
with open("documents/semantic_chunks_with_metadata.json", "r") as f:
    sample_data = json.load(f)
file_name="documents/pdfdata\\Constitution_India.pdf"
count=0
n=int(input(f"Enter Chunk number to retrieve for {file_name}: "))
for item in sample_data:
    if item['metadata']['source']==file_name:
        count+=1
        if count==n:
            print(f"Found matching item for {file_name}:")
            print(f"Metadata: {item['metadata']}")
            print(f"Content Preview: {str(item['content'])[:1000]}") # Print the first 1000 characters of content
            break
        

Found matching item for documents/pdfdata\Constitution_India.pdf:
Metadata: {'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2024-07-01T11:20:33+00:00', 'source': 'documents/pdfdata\\Constitution_India.pdf', 'total_pages': 402, 'page': 50, 'page_label': '51'}
Content Preview: 35. Legislation to give effect to the provisions of this Part.—Notwithstanding anything in this Constitution,—  (a) Parliament shall have, and the Legislature of a State shall not have, power to make laws—(i) with respect to any of the matters which under clause (3) of article 16, clause (3) of article 32, article 33 and article 34 may be provided for by law made by Parliament; and (ii) for prescribing punishment for those acts which are declared to be offences under this Part,and Parliament shall, as soon as may be after the commencement of this Constitution, make laws for prescribing punishment for the acts referred to in sub-clause (ii);(b) any law in force immediately before the com

In [10]:
import json
import chromadb
from sentence_transformers import SentenceTransformer

model=SentenceTransformer("all-MiniLM-L6-v2")
client=chromadb.PersistentClient(path="./chroma_storage")
collection=client.get_or_create_collection(name="legal_reports")
def migrate_chunks_to_db():
    with open("documents/semantic_chunks_with_metadata.json","r") as fp:
        chunks=json.load(fp)
    print(f'Vectorizing and inserting {len(chunks)} chunks into ChromaDB...')
    for i,item in enumerate(chunks):
        content=item.get('content') 
        metadata=item.get('metadata',{})
        collection.add(documents=[content],
                       metadatas=[metadata],
                       ids=[f"chunk_{i}"])
        if(i%50==0):
            print(f"Processed {i} chunks so far...")
    print("All chunks migrated to ChromaDB!")
migrate_chunks_to_db()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6617.55it/s]


Vectorizing and inserting 867 chunks into ChromaDB...
Processed 0 chunks so far...
Processed 50 chunks so far...
Processed 100 chunks so far...
Processed 150 chunks so far...
Processed 200 chunks so far...
Processed 250 chunks so far...
Processed 300 chunks so far...
Processed 350 chunks so far...
Processed 400 chunks so far...
Processed 450 chunks so far...
Processed 500 chunks so far...
Processed 550 chunks so far...
Processed 600 chunks so far...
Processed 650 chunks so far...
Processed 700 chunks so far...
Processed 750 chunks so far...
Processed 800 chunks so far...
Processed 850 chunks so far...
All chunks migrated to ChromaDB!


In [12]:
from pprint import pprint
test_query="Powers of the President of India"
results=collection.query(query_texts=[test_query],n_results=3)
pprint(results)
#pprint(results['metadatas'][0])


{'data': None,
 'distances': [[0.6523882746696472, 0.6989151835441589, 0.7019559741020203]],
 'documents': [['26\n'
                'PART  VTHE UNIONCHAPTER I.—THE EXECUTIVEThe President and '
                'Vice-President52. The President of India.—There shall be a '
                'President of India.53. Executive power of the Union.—(1) The '
                'executive power of the Union shall be vested in the President '
                'and shall be exercised by him either directly or through '
                'officers subordinate to him in accordance with this '
                'Constitution.(2) Without prejudice to the generality of the '
                'foregoing provision, the supreme command of the Defence '
                'Forces of the Union shall be vested in the President and the '
                'exercise thereof shall be regulated by law.(3) Nothing in '
                'this article shall— (a) be deemed to transfer to the '
                'President any functio

In [13]:
print("Top 3 relevant chunks from ChromaDB:")
for i, res in enumerate(results['documents'][0]):
    print(f"\n--- Chunk {i+1} ---")
    print(res[:1000]) # Print the first 1000 characters of each chunk

Top 3 relevant chunks from ChromaDB:

--- Chunk 1 ---
26
PART  VTHE UNIONCHAPTER I.—THE EXECUTIVEThe President and Vice-President52. The President of India.—There shall be a President of India.53. Executive power of the Union.—(1) The executive power of the Union shall be vested in the President and shall be exercised by him either directly or through officers subordinate to him in accordance with this Constitution.(2) Without prejudice to the generality of the foregoing provision, the supreme command of the Defence Forces of the Union shall be vested in the President and the exercise thereof shall be regulated by law.(3) Nothing in this article shall— (a) be deemed to transfer to the President any functions conferred by any existing law on the Government of any State or other authority; or(b) prevent Parliament from conferring by law functions on authorities other than the President. 54. Election of President.—The President shall be elected by the members of an electoral college consi

In [14]:
import os
from openai import OpenAI
from dotenv import load_dotenv


In [15]:
load_dotenv() # Load environment variables from .env file
client=OpenAI(
    api_key=os.getenv("GROQ_API_KEY") ,# Ensure you have GROQ_API_KEY set in your .env file
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
def legal_rag_query(user_query):
    results = collection.query(
        query_texts=[user_query],
        n_results=15,
        )
    retrieved_chunks=results['documents'][0]

    if not retrieved_chunks:
        return "Target file not found or no relevant chunks found."
    
    context_block="\n\n".join(retrieved_chunks)
    #print(context_block[:2000]) # Print the first 2000 characters of the context for debugging
    system_prompt = f"""
    You are an expert Legal Scholar Agent specializing in the Constitution of India. 
    Your goal is to answer questions based ONLY on the provided legal text context.
    
    RULES:
    1. Only use the provided CONTEXT to answer. Do not bring in outside legal knowledge or assumptions.
    2. If the exact answer or relevant article is not explicitly present in the provided context, say: "I'm sorry, my local vault does not contain specific data to answer this."
    3. Always explicitly cite the relevant Article number, Clause, and Part of the Constitution if mentioned in the context (e.g., "According to Article 21 under Part III...").
    4. Always mention the source file name if available in the metadata.
    
    CONTEXT:
    {context_block}
    """
    chat_completion=client.chat.completions.create(
        messages=[{"role":"system","content":system_prompt},{"role":"user","content":user_query}],
        model="llama-3.3-70b-versatile",
        temperature=0.1
    )
    return chat_completion.choices[0].message.content.strip()
query = "Fundamental Rights guaranteed by the Constitution of India and their significance in modern times."
response = legal_rag_query(query)
print("Legal RAG Response:")
print(response)
'''debug_results=collection.query(query_texts=["revenue outlook guidance 2026"], n_results=20)
for i, text in enumerate(debug_results['documents'][0]):
    print(f"CHUNK {i} | Source: {debug_results['metadatas'][0][i].get('source')} | Page: {debug_results['metadatas'][0][i].get('page')}")
    print(f"Snippet: {text[:200]}...")
    print("-" * 30)'''

Legal RAG Response:
According to the Constitution of India (Part III.—Fundamental Rights), the Fundamental Rights guaranteed by the Constitution of India are significant in modern times as they provide a framework for the protection of individual rights and freedoms. 

As stated in Article 14 under Part III, "The State shall not deny to any person equality before the law or the equal protection of the laws within the territory of India." This article emphasizes the importance of equality before the law, ensuring that all individuals are treated equally and without discrimination.

Article 15 under Part III prohibits discrimination on grounds of religion, race, caste, sex, or place of birth. This article is crucial in modern times as it promotes equality and prevents discrimination against marginalized groups.

Article 19 under Part III guarantees the right to freedom of speech and expression, which is essential in modern times for the functioning of a democratic society. 

Article 21 u

'debug_results=collection.query(query_texts=["revenue outlook guidance 2026"], n_results=20)\nfor i, text in enumerate(debug_results[\'documents\'][0]):\n    print(f"CHUNK {i} | Source: {debug_results[\'metadatas\'][0][i].get(\'source\')} | Page: {debug_results[\'metadatas\'][0][i].get(\'page\')}")\n    print(f"Snippet: {text[:200]}...")\n    print("-" * 30)'

: 